In [ ]:
# 1. 아두이노에서 랜덤으로 주문 코드 생성(기본 주문이 들어가고 난후, 결제 전에는 추가 주문 생성)(아두이노 디스플레이에는 눌렀을 때 주문 표시)
# 2. 주문 코드를 주문 리스트에 저장
# 3. 기능: 매장 구조에 맞는 테이블에 주문 텍스트 생성, 주문 순서(추가 주문), 결제, 총 매출액, 받을금액, 받은금액, 예약 리스트
# 4. 상시로 표시될 기능 - 테이블 및 주문 표시, 총 매출액, 결제 메뉴, 예약리스트 메뉴, 주문순서 메뉴 - 함수 정의
# 5. 주문 관리, 결제 완료 리스트 추가 - class로 구현
print(f"┌{"─" * 7}┐")
print("│  │")
빨22(24) 초21(22)   빨12(39) 초11(37)   초2(35) 검1(33)
갈24(28) 주23(26)   초14(36) 파13(34)   초4(31) 파3(29)
노26(32) 검25(30)   흰16(40) 흰15(38)   초6(27) 파5(25)
                                       흰8(41) 보7(23)

┌───────┐
│ 야채 사리 추가 │


In [ ]:
while True:
	print(f"{"=" * }")

In [ ]:
#include <MCUFRIEND_kbv.h>
#include <Adafruit_GFX.h>

#define NUM_BUTTONS 20

const int buttonPins[NUM_BUTTONS] = {
  22,23,24,25,26,27,28,29,30,31,
  32,33,34,35,36,37,38,39,40,41
};

const int buttonNum[NUM_BUTTONS] = {
  21,7,22,5,23,6,24,3,25,4,
  26,1,13,2,14,11,15,12,16,8
};

struct OrderItem {
  const char* code;
  const char* name;
};

OrderItem orders[] = {
  {"A101", "KIMCHI JJIGAE"},
  {"A102", "BIBIMBAP"},
  {"A103", "DONKKASEU"},
  {"A104", "RAMEN"},
  {"A105", "TTEOKBOKKI"},
  {"A106", "UDON"},
  {"A107", "JAEYUK DEOPBAP"},
  {"A108", "SOY GARLIC CHICKEN"},
  {"A109", "BULGOGI DEOPBAP"},
  {"A110", "CHEESE DONKKASEU"}
};

const int ORDER_COUNT = sizeof(orders) / sizeof(orders[0]);

MCUFRIEND_kbv tft;
uint16_t lcdID;

int lastStableState[NUM_BUTTONS];
int lastReading[NUM_BUTTONS];
unsigned long lastDebounceTime[NUM_BUTTONS];
const unsigned long debounceDelay = 50;

String lastOrderCode = "";
String lastOrderName = "";

void drawCenteredText(const char* txt, int y, int textSize, uint16_t color) {
  int16_t x1, y1;
  uint16_t w, h;
  tft.setTextSize(textSize);
  tft.setTextColor(color);
  tft.getTextBounds((char*)txt, 0, y, &x1, &y1, &w, &h);
  int x = (tft.width() - w) / 2;
  if (x < 0) x = 0;
  tft.setCursor(x, y);
  tft.print(txt);
}

void drawScreenFrame() {
  tft.fillScreen(0x0000);
  tft.drawRect(0, 0, tft.width(), tft.height(), 0xFFFF);
  tft.drawFastHLine(0, 36, tft.width(), 0xFFFF);
}

void showReadyScreen() {
  drawScreenFrame();
  drawCenteredText("ORDER SIMULATOR", 10, 2, 0xFFFF);
  drawCenteredText("PRESS ANY ORDER BUTTON", 70, 2, 0x07FF);
  drawCenteredText("BTN 20 = DELETE", 110, 2, 0xF800);
}

void showOrderOnTFT(const char* code, const char* name, int sourceButton) {
  drawScreenFrame();
  drawCenteredText("NEW ORDER", 10, 2, 0xFFE0);

  tft.setTextSize(2);
  tft.setTextColor(0xFFFF);
  tft.setCursor(10, 55);
  tft.print("BTN: ");
  tft.print(buttonNum[sourceButton-1]);

  tft.setCursor(140, 55);
  tft.print("CODE: ");
  tft.print(code);

  drawCenteredText(name, 100, 3, 0x07E0);
}

void showDeleteOnTFT() {
  drawScreenFrame();
  drawCenteredText("DELETE", 30, 3, 0xF800);

  tft.setTextSize(2);
  tft.setTextColor(0xFFFF);
  tft.setCursor(20, 95);
  tft.print("LAST ORDER REMOVED");

  if (lastOrderCode.length() > 0) {
    tft.setCursor(20, 130);
    tft.print("CODE: ");
    tft.print(lastOrderCode);

    tft.setCursor(20, 160);
    tft.print("NAME: ");
    tft.print(lastOrderName);
  }
}

void sendRandomOrder(int sourceButton) {
  int idx = random(0, ORDER_COUNT);

  lastOrderCode = orders[idx].code;
  lastOrderName = orders[idx].name;

  Serial.print("ORDER,");
  Serial.print(orders[idx].code);
  Serial.print(",");
  Serial.print(sourceButton);
  Serial.print(",");
  Serial.println(buttonNum[sourceButton - 1]);

  showOrderOnTFT(orders[idx].code, orders[idx].name, sourceButton);
}

void sendDeleteCommand() {
  Serial.println("DELETE");

  if (lastOrderCode.length() > 0) {
    showDeleteOnTFT();
    lastOrderCode = "";
    lastOrderName = "";
  } else {
    drawScreenFrame();
    drawCenteredText("DELETE", 30, 3, 0xF800);
    drawCenteredText("NO ORDER TO REMOVE", 110, 2, 0xFFFF);
  }
}

void setup() {
  Serial.begin(9600);

  for (int i = 0; i < NUM_BUTTONS; i++) {
    pinMode(buttonPins[i], INPUT_PULLUP);
    lastStableState[i] = HIGH;
    lastReading[i] = HIGH;
    lastDebounceTime[i] = 0;
  }

  randomSeed(analogRead(A15));

  lcdID = tft.readID();
  if (lcdID == 0xD3D3) lcdID = 0x9486;
  tft.begin(lcdID);

  tft.setRotation(1);
  tft.fillScreen(0x0000);

  showReadyScreen();
}

void loop() {
  for (int i = 0; i < NUM_BUTTONS; i++) {
    int reading = digitalRead(buttonPins[i]);

    if (reading != lastReading[i]) {
      lastDebounceTime[i] = millis();
      lastReading[i] = reading;
    }

    if ((millis() - lastDebounceTime[i]) > debounceDelay) {
      if (reading != lastStableState[i]) {
        lastStableState[i] = reading;

        if (lastStableState[i] == LOW) {
          if (i == NUM_BUTTONS - 1) {
            sendDeleteCommand();
          } else {
            sendRandomOrder(i + 1);
          }
        }
      }
    }
  }
}